In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cvxpy as cp
import os
from hmmlearn.hmm import GaussianHMM
from dataclasses import dataclass, asdict


In [ ]:
mkt_sectors = [
  "XLK",
  "XLF",
  "XLE",
  "XLV",
  "XLU"
]

In [4]:
class DataStore:
  def __init__(self, debug:bool=False, **kwargs):
    self.debug = debug

  def download(self, indices:str, start:str, end:str, interval:str):
    data = {}
    for ticker in indices:
      clean_ticker = ticker.strip()
      try:
        tmp_data = yf.Ticker(clean_ticker) \
                      .history(
                        start=start,
                        end=end,
                        interval=interval
                      )["Close"]

        if tmp_data.empty:
          print(f"Warning: No data returned for {clean_ticker}")
          continue

        tmp_data.index = pd.to_datetime(tmp_data.index)\
                                .tz_localize(None)\
                                .normalize()
        tmp_data = tmp_data[~tmp_data.index.duplicated(keep='last')]
        data[clean_ticker] = tmp_data
      except Exception as e:
        print(f"Failed to download {clean_ticker}: {e}")

    if not data:
      raise ValueError(
        "No data was successfully downloaded  \
          for any of the provided tickers."
      )
    df = pd.concat(data, axis=1)
    return df.sort_index()

  def get_data(
    self,
    start:str,
    end:str,
    mkt_sectors:list[str],
    interval:str="1d"
  ):
    sector_path = f"sector_{start}_{end}.parquet"

    if not os.path.exists(sector_path):
      mkt_sectors_data = self.download(mkt_sectors, start, end, interval)
      mkt_sectors_data.to_parquet(sector_path)

    else:
      mkt_sectors_data= pd.read_parquet(sector_path)

    benchmark = mkt_sectors_data.pct_change().dropna()

    return benchmark

  def get_market_data(self, ticker, start, end, interval="1d"):
    return yf.download(ticker, start=start, end=end, interval=interval)

  def plot_data(self):
    (np.cumsum(self.returns_raw * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

  def plot_benchmark(self):
    (np.cumsum(self.benchmark * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

In [18]:
class HMMClassifier:
  def __init__(self, hmm_params, debug=False, **kwargs):
    self.debug = debug
    hmm_params = asdict(hmm_params) if hasattr(hmm_params, '__dataclass_fields__') else hmm_params
    self.hmm = GaussianHMM(**hmm_params)
    self.trained = False

  def fit(self, X):
    self.trained = True
    return self.hmm.fit(X)

  def get_regime_probs(self, X):
    if not self.trained:
      raise ValueError("HMM Classifier not trained")

    return self.hmm.predict_proba(X)

In [19]:
class RegimeClassifier(HMMClassifier):
  def __init__(self, hmm_params, debug=False, **kwargs):
    super().__init__(hmm_params=hmm_params, debug=debug, **kwargs)
    self.s_mean = None
    self.s_std = None
    self.eff_dim_mean = None
    self.eff_dim_std = None
    self.vol_mean = None
    self.vol_std = None

  def _apply_ledoit_wolf(self, r_norm, gram_sample, T, n_assets):
    mean_mkt_corr = (np.sum(gram_sample) - n_assets) / (n_assets * (n_assets - 1))
    target = np.full((n_assets, n_assets), mean_mkt_corr)
    np.fill_diagonal(target, 1.0)

    noise_mtx = 0.0
    for t in range(T):
      r_t = r_norm[t, :].reshape(-1, 1)
      sample_t_mtx = r_t @ r_t.T
      noise_mtx += np.sum((sample_t_mtx - gram_sample) ** 2)

    noise = noise_mtx / (T ** 2)
    dist  = np.sum((gram_sample - target) ** 2)

    if dist == 0:
      return gram_sample

    shrinkage_intensity = np.clip(noise / dist, 0.0, 1.0)
    return shrinkage_intensity * target + (1 - shrinkage_intensity) * gram_sample

  def calculate_spec_ent(self, eig_vals):
    regime_probs = eig_vals / np.sum(eig_vals)
    regime_probs = np.clip(regime_probs, 1e-12, 1.0)
    return -np.sum(regime_probs * np.log(regime_probs))

  def _get_eff_dim(self, r_norm, T):
    G = (1 / T) * (r_norm.T @ r_norm)
    n_assets = G.shape[0]
    G_cond   = self._apply_ledoit_wolf(r_norm, G, T, n_assets)
    eig_vals = np.clip(np.linalg.eigvalsh(G_cond), a_min=1e-12, a_max=None)
    spec_ent = self.calculate_spec_ent(eig_vals)
    return np.exp(spec_ent)

  def garman_klass_vol(self, market, lambda_param=0.94):
    ln_CO = np.log(market["Close"] / market["Open"])
    ln_HL = np.log(market["High"] / market["Low"])
    gk_var = 0.5 * (ln_HL ** 2) - (2 * np.log(2) - 1) * (ln_CO ** 2)

    T = len(gk_var)
    smoothed_variance    = np.zeros(T)
    smoothed_variance[0] = gk_var.iloc[0]
    for t in range(1, T):
      smoothed_variance[t] = (lambda_param * smoothed_variance[t - 1]) + ((1 - lambda_param) * gk_var.iloc[t])
    return np.sqrt(smoothed_variance * 252)

  def _get_rolling_eff_dim(self, r_norm, lookback: int = 60):
      T, n_assets = r_norm.shape
      eff_dim_series = np.full(T, np.nan)
      min_obs = max(10, n_assets + 2)

      for t in range(T):
        start = max(0, t - lookback + 1)
        window = r_norm[start:t + 1, :]
        if window.shape[0] < min_obs:
          continue
        eff_dim_series[t] = self._get_eff_dim(window, window.shape[0])

      valid_mask = ~np.isnan(eff_dim_series)
      if not valid_mask.any():
        raise ValueError("Not enough observations to compute effective dimension.")

      first_valid = np.argmax(valid_mask)
      eff_dim_series[:first_valid] = eff_dim_series[first_valid]
      return eff_dim_series

  def fit_transform_features(self, sector_returns, market_ohlc, eff_dim_lookback=60):
    s_r_log = np.log(1.0 + sector_returns).values

    self.s_mean = np.mean(s_r_log, axis=0)
    self.s_std = np.std(s_r_log, axis=0, ddof=0)
    self.s_std[self.s_std == 0] = 1.0

    s_r_norm = (s_r_log - self.s_mean) / self.s_std
    eff_dim_series = self._get_rolling_eff_dim(s_r_norm, lookback=eff_dim_lookback)
    vol = self.garman_klass_vol(market_ohlc)

    self.eff_dim_mean, self.eff_dim_std = np.mean(eff_dim_series), np.std(eff_dim_series, ddof=0)
    self.vol_mean, self.vol_std = np.mean(vol), np.std(vol, ddof=0)

    eff_dim_norm = (eff_dim_series - self.eff_dim_mean) / (self.eff_dim_std if self.eff_dim_std > 0 else 1.0)
    vol_norm = (vol - self.vol_mean) / (self.vol_std if self.vol_std > 0 else 1.0)

    return np.column_stack((eff_dim_norm, vol_norm))

  def transform_features(self, sector_returns, market_ohlc, eff_dim_lookback=60):
    if self.s_mean is None:
        raise ValueError("Classifier must be trained before calling transform.")

    s_r_log = np.log(1.0 + sector_returns).values
    s_r_norm = (s_r_log - self.s_mean) / self.s_std
    eff_dim_series = self._get_rolling_eff_dim(s_r_norm, lookback=eff_dim_lookback)
    vol = self.garman_klass_vol(market_ohlc)

    eff_dim_norm = (eff_dim_series - self.eff_dim_mean) / (self.eff_dim_std if self.eff_dim_std > 0 else 1.0)
    vol_norm = (vol - self.vol_mean) / (self.vol_std if self.vol_std > 0 else 1.0)

    return np.column_stack((eff_dim_norm, vol_norm))

  def train_classifier(self, returns, market):
    X = self.fit_transform_features(returns, market)
    self.fit(X)

  def get_regime_probs(self, returns, market):
    X = self.transform_features(returns, market)
    return super().get_regime_probs(X)

  def score_bic(self, X):
    T, n_features  = X.shape
    n_components   = self.hmm.n_components
    log_likelihood = self.hmm.score(X) * T
    k = n_components * (n_components - 1) + 2 * (n_components * n_features)
    return k * np.log(T) - 2 * log_likelihood

  def expected_regime_duration(self):
    diag = np.clip(np.diag(self.hmm.transmat_), 1e-10, 1.0 - 1e-4)
    return 1.0 / (1.0 - diag)

In [25]:
class Backtest:
  def __init__(
    self,
    regime_classifier_cls,
    hmm_params,
    train_months: int = 12,
    test_months: int = 6,
    step_months: int = 6,
    min_train_obs: int = 30,
    min_test_obs: int = 5,
    debug: bool = False,
  ):
    self.regime_classifier_cls = regime_classifier_cls
    self.hmm_params = hmm_params
    self.train_months = train_months
    self.test_months = test_months
    self.step_months = step_months
    self.min_train_obs = min_train_obs
    self.min_test_obs = min_test_obs
    self.debug = debug
    self.results = []

  def _generate_windows(self, index):
    windows = []
    data_end = index[-1]
    train_start = index[0]

    while True:
      train_end = train_start + pd.DateOffset(months=self.train_months)
      test_end = train_end + pd.DateOffset(months=self.test_months)
      if test_end > data_end:
        break
      windows.append((train_start, train_end, test_end))
      train_start = train_start + pd.DateOffset(months=self.step_months)

    return windows

  def run_regime_backtest(self, sector_returns: pd.DataFrame, market_returns: pd.Series):
    self.results = []
    windows = self._generate_windows(sector_returns.index)

    if not windows:
      raise ValueError(
        "Not enough history to build a single walk-forward window "
        f"(need >= {self.train_months + self.test_months} months)."
      )

    for i, (train_start, train_end, test_end) in enumerate(windows):
      train_sector = sector_returns.loc[train_start:train_end]
      train_market = market_returns.loc[train_start:train_end]
      test_sector  = sector_returns.loc[train_end:test_end]
      test_market  = market_returns.loc[train_end:test_end]

      if len(train_sector) < self.min_train_obs or len(test_sector) < self.min_test_obs:
        if self.debug:
          print(
            f"Window {i}: insufficient data, skipping "
            f"(train={len(train_sector)}, test={len(test_sector)})"
          )
        continue

      classifier = self.regime_classifier_cls(hmm_params=self.hmm_params, debug=self.debug)

      if i > 0 and len(self.results) > 0 and hasattr(self, 'prev_hmm_'):
        classifier.hmm.transmat_ = self.prev_hmm_.transmat_.copy()
        classifier.hmm.means_ = self.prev_hmm_.means_.copy()
        classifier.hmm.covars_ = self.prev_hmm_._covars_.copy()
        classifier.hmm.startprob_ = self.prev_hmm_.startprob_.copy()
        classifier.hmm.init_params = ""

      try:
        classifier.train_classifier(train_sector, train_market)
        self.prev_hmm_ = classifier.hmm
      except Exception as e:
        if self.debug:
          print(f"Window {i}: training failed ({e}), skipping")
        continue

      X_train = classifier.fit_transform_features(train_sector, train_market)
      bic = classifier.score_bic(X_train)
      expected_duration = classifier.expected_regime_duration()

      high_vol_state = int(np.argmax(classifier.hmm.means_[:, 1]))

      try:
        test_probs = classifier.get_regime_probs(test_sector, test_market)
        test_states = np.argmax(test_probs, axis=1)
      except Exception as e:
        if self.debug:
          print(f"Window {i}: OOS scoring failed ({e}), skipping test stats")
        test_probs, test_states = None, None

      window_result = {
        "window": i,
        "train_start": train_start,
        "train_end": train_end,
        "test_end": test_end,
        "n_train": len(train_sector),
        "n_test": len(test_sector),
        "bic": bic,
        "expected_duration": expected_duration,
        "high_vol_state": high_vol_state,
        "transmat": classifier.hmm.transmat_.copy(),
        "test_state_probs": test_probs,
        "test_states": test_states,
      }
      self.results.append(window_result)

      if self.debug:
        dur_str = np.round(expected_duration, 2)
        print(
          f"Window {i}: {train_start.date()} -> {train_end.date()} -> {test_end.date()} "
          f"| BIC={bic:.2f} | high_vol_state={high_vol_state} | durations={dur_str}"
        )

    return self.summary()

  def summary(self) -> pd.DataFrame:
    if not self.results:
      return pd.DataFrame()

    rows = []
    for r in self.results:
      row = {
        "window": r["window"],
        "train_start": r["train_start"],
        "train_end": r["train_end"],
        "test_end": r["test_end"],
        "n_train": r["n_train"],
        "n_test": r["n_test"],
        "bic": r["bic"],
        "high_vol_state": r["high_vol_state"],
      }
      for state_idx, dur in enumerate(r["expected_duration"]):
        row[f"exp_duration_state_{state_idx}"] = dur
      rows.append(row)

    return pd.DataFrame(rows).set_index("window")

  def regime_path(self) -> pd.Series:
    pieces = []
    for r in self.results:
      if r["test_states"] is None:
        continue
      idx = pd.date_range(r["train_end"], r["test_end"], periods=len(r["test_states"]))
      pieces.append(pd.Series(r["test_states"], index=idx))
    if not pieces:
      return pd.Series(dtype=int)
    return pd.concat(pieces).sort_index()

In [31]:
@dataclass
class HMMParams:
    n_components: int = 2
    n_iter: int = 50
    tol: float = 1e-4
    init_params: str = "stmc"
    covariance_type: str = "diag"

store = DataStore(debug=False)
benchmark = store.get_data(
    start="2015-01-01",
    end="2025-01-01",
    mkt_sectors=["XLK", "XLF", "XLE", "XLV", "XLY"],
)

market_ohlc = store.get_market_data(
    ticker="SPY",
    start="2015-01-01",
    end="2025-01-01",
)

hmm_params = HMMParams(n_components=2, n_iter=100, covariance_type="diag")

bt = Backtest(
    regime_classifier_cls=RegimeClassifier,
    hmm_params=hmm_params,
    train_months=12,
    test_months=6,
    step_months=6,
    debug=True,
)

results_summary = bt.run_regime_backtest(sector_returns=benchmark, market_returns=market_ohlc)
print(results_summary)

regime_series = bt.regime_path()

/tmp/ipykernel_711/2290980758.py:58: FutureWarning: YF.download() has changed argument auto_adjust default to True
  return yf.download(ticker, start=start, end=end, interval=interval)
[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_711/2328335496.py:51: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  smoothed_variance[0] = gk_var.iloc[0]
/tmp/ipykernel_711/2328335496.py:53: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  smoothed_variance[t] = (lambda_param * smoothed_variance[t - 1]) + ((1 - lambda_param) * gk_var.iloc[t])
/tmp/ipykernel_711/2328335496.py:51: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  smoothed_variance[0] = gk_var.iloc[0]
/tmp/ipykernel_711/

Window 0: 2015-01-05 -> 2016-01-05 -> 2016-07-05 | BIC=134004.48 | high_vol_state=1 | durations=[77.12 32.63]


/tmp/ipykernel_711/2328335496.py:51: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  smoothed_variance[0] = gk_var.iloc[0]
/tmp/ipykernel_711/2328335496.py:53: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  smoothed_variance[t] = (lambda_param * smoothed_variance[t - 1]) + ((1 - lambda_param) * gk_var.iloc[t])
/tmp/ipykernel_711/2328335496.py:51: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  smoothed_variance[0] = gk_var.iloc[0]
/tmp/ipykernel_711/2328335496.py:53: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  smoothed_variance[t] = (lambda_param * smoothed_variance[t - 1]) + ((1 - lambda_param) * gk

Window 1: 2015-07-05 -> 2016-07-05 -> 2017-01-05 | BIC=162582.47 | high_vol_state=1 | durations=[63.42 41.71]


/tmp/ipykernel_711/2328335496.py:51: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  smoothed_variance[0] = gk_var.iloc[0]
/tmp/ipykernel_711/2328335496.py:53: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  smoothed_variance[t] = (lambda_param * smoothed_variance[t - 1]) + ((1 - lambda_param) * gk_var.iloc[t])
/tmp/ipykernel_711/2328335496.py:51: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  smoothed_variance[0] = gk_var.iloc[0]
/tmp/ipykernel_711/2328335496.py:53: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  smoothed_variance[t] = (lambda_param * smoothed_variance[t - 1]) + ((1 - lambda_param) * gk

...

/tmp/ipykernel_711/2328335496.py:51: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  smoothed_variance[0] = gk_var.iloc[0]
/tmp/ipykernel_711/2328335496.py:53: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  smoothed_variance[t] = (lambda_param * smoothed_variance[t - 1]) + ((1 - lambda_param) * gk_var.iloc[t])
/tmp/ipykernel_711/2328335496.py:51: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  smoothed_variance[0] = gk_var.iloc[0]
/tmp/ipykernel_711/2328335496.py:53: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  smoothed_variance[t] = (lambda_param * smoothed_variance[t - 1]) + ((1 - lambda_param) * gk

Window 14: 2022-01-05 -> 2023-01-05 -> 2023-07-05 | BIC=277614.57 | high_vol_state=1 | durations=[38.09 45.39]


/tmp/ipykernel_711/2328335496.py:51: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  smoothed_variance[0] = gk_var.iloc[0]
/tmp/ipykernel_711/2328335496.py:53: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  smoothed_variance[t] = (lambda_param * smoothed_variance[t - 1]) + ((1 - lambda_param) * gk_var.iloc[t])
/tmp/ipykernel_711/2328335496.py:51: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  smoothed_variance[0] = gk_var.iloc[0]
/tmp/ipykernel_711/2328335496.py:53: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  smoothed_variance[t] = (lambda_param * smoothed_variance[t - 1]) + ((1 - lambda_param) * gk

Window 15: 2022-07-05 -> 2023-07-05 -> 2024-01-05 | BIC=207256.49 | high_vol_state=1 | durations=[62.6  62.71]
Window 16: 2023-01-05 -> 2024-01-05 -> 2024-07-05 | BIC=223872.35 | high_vol_state=1 | durations=[56.51 68.93]
       train_start  train_end   test_end  n_train  n_test            bic  \
window                                                                     
0       2015-01-05 2016-01-05 2016-07-05      253     126  134004.477635   
1       2015-07-05 2016-07-05 2017-01-05      253     129  162582.471752   
2       2016-01-05 2017-01-05 2017-07-05      254     125  167884.281277   
3       2016-07-05 2017-07-05 2018-01-05      253     129  213895.996709   
4       2017-01-05 2018-01-05 2018-07-05      253     125  299097.931665   
5       2017-07-05 2018-07-05 2019-01-05      253     127   37022.597977   
6       2018-01-05 2019-01-05 2019-07-05      251     125  146682.578144   
7       2018-07-05 2019-07-05 2020-01-05      252     127  116726.636630   
8       2019-01-05

/tmp/ipykernel_711/2328335496.py:51: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  smoothed_variance[0] = gk_var.iloc[0]
/tmp/ipykernel_711/2328335496.py:53: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  smoothed_variance[t] = (lambda_param * smoothed_variance[t - 1]) + ((1 - lambda_param) * gk_var.iloc[t])
/tmp/ipykernel_711/2328335496.py:51: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  smoothed_variance[0] = gk_var.iloc[0]
/tmp/ipykernel_711/2328335496.py:53: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  smoothed_variance[t] = (lambda_param * smoothed_variance[t - 1]) + ((1 - lambda_param) * gk